# RAG 기초 (영어 문서) — LangChain + 로컬 임베딩

`rag_basic_ko.ipynb` 와 같은 흐름이지만 세 가지가 다릅니다.

| | `rag_basic_ko` | 이 노트북 |
|---|---|---|
| 문서 | 사내 보안 가이드 (한글, 직접 작성) | **이상한 나라의 앨리스** (영어, 인터넷에서 내려받기) |
| 임베딩 | `OpenAIEmbeddings` (유료 API) | **`HuggingFaceEmbeddings`** (내 PC, 공짜) |
| 분량 | 짧은 문서 | 책 한 권 (약 14만 자) |

## 흐름

```
[준비] 책 내려받기 → 청킹 → 임베딩 → Chroma 에 저장
[질문] 질문 → 비슷한 청크 검색 → 질문 + 청크를 LLM 에 → 답변
```

## 이 노트북에서 꼭 볼 것 두 가지

| | 어디서 | 무엇을 |
|---|---|---|
| **검색이 무엇을 돌려줬는지 확인** | Step 14 | 답이 맞아 보여도 검색은 망가져 있을 수 있습니다 |
| **청크가 답을 자르면** | Step 18 | 같은 책·같은 질문인데 청크 크기만 바꿔도 답이 달라집니다 |

> 둘 다 **답변만 읽어서는 절대 알 수 없는** 문제입니다.

## Step 1. 기본 준비

`tiktoken` 으로 토큰을 세고(`ntok`), `OpenAI` 클라이언트를 만듭니다.
`1_5_tokenizer.ipynb` 에서 쓰던 것과 같습니다.

> `cl100k_base` 는 GPT-4 계열 토크나이저입니다.
> 뒤에서 쓰는 임베딩 모델(`all-MiniLM-L6-v2`)은 토크나이저가 다르므로,
> 여기서 센 토큰 수는 **대략적인 분량 감**으로만 보세요.

In [5]:
import re

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

## Step 2. LangChain 부품 가져오기

여기서부터 LangChain 을 씁니다. 각 부품이 맡는 일이 분명합니다.

| 부품 | 하는 일 |
|---|---|
| `RecursiveCharacterTextSplitter` | 문서를 청크로 자른다 |
| `HuggingFaceEmbeddings` | 글을 벡터로 바꾼다 — **내 PC 에서** |
| `Chroma` | 벡터를 저장하고 검색한다 |
| `DirectoryLoader`, `TextLoader` | 폴더·파일에서 문서를 읽는다 (이 노트북에서는 쓰지 않음) |
| `TSNE` | 벡터를 2차원으로 눌러 그림으로 보는 도구 (이 노트북에서는 쓰지 않음) |

> `chunk_basic_ko1.ipynb` 에서는 `numpy` 행렬과 내적으로 직접 검색했습니다.
> 청크가 수백 개일 때는 그걸로 충분했지만, 책 한 권이면 DB 가 편합니다.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
#import plotly.graph_objects as go

## Step 3. 책 내려받기

Project Gutenberg 에서 **이상한 나라의 앨리스** 전문을 받습니다.

```python
start = raw.index("*** START")
end = raw.index("*** END")
book = raw[raw.index("\n", start) + 1 : end].strip()
```

Gutenberg 파일에는 본문 앞뒤로 **저작권 안내문**이 길게 붙어 있습니다.
`*** START` 와 `*** END` 사이만 잘라내는 이유입니다. 안 자르면 법률 문구가 청크에 섞입니다.

> **주소를 두 개 준비하고 재시도합니다.**
> Gutenberg 는 접속이 몰리면 `503` 을 돌려줍니다. 실제로 만드는 동안에도 자주 막혔습니다.
> 한 번에 성공하는 코드보다, **실패를 견디는 코드**가 실습에서는 더 쓸모 있습니다.

> `book` 은 **문자열(`str`)** 입니다. 리스트가 아닙니다. Step 7 에서 이 점이 중요해집니다.

In [7]:
import time

# Gutenberg 는 접속이 몰리면 503 을 돌려줍니다. 주소 두 개를 번갈아 재시도합니다.
URLS = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",   # Alice in Wonderland
    "https://www.gutenberg.org/files/11/11-0.txt",
]

raw = None
for attempt in range(1, 6):
    for url in URLS:
        try:
            resp = requests.get(url, timeout=45)
            if resp.status_code == 200 and len(resp.text) > 100_000:
                resp.encoding = "utf-8"
                raw = resp.text
                break
        except Exception as err:
            print(f"  시도 {attempt}: {type(err).__name__}")
    if raw:
        break
    print(f"  시도 {attempt} 실패, 잠시 뒤 다시 시도합니다…")
    time.sleep(5)

if raw is None:
    raise RuntimeError("Gutenberg 에서 책을 받지 못했습니다. 잠시 뒤 다시 실행하세요.")

# Strip Project Gutenberg's legal header and footer
start = raw.index("*** START")
end = raw.index("*** END")
book = raw[raw.index("\n", start) + 1 : end].strip()

print(f"{len(book):,} characters   {ntok(book):,} tokens")
print(f"{len(re.findall(r'^CHAPTER [IVX]+', book, re.M))} chapters")
print()
print(book[:1000])

144,599 characters   36,926 tokens
12 chapters

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her 

## Step 4. 청킹

```python
RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_text(book)
```

| 인자 | 뜻 |
|---|---|
| `chunk_size=1000` | 청크 하나를 약 1,000자로 |
| `chunk_overlap=200` | 앞 청크의 끝 200자를 다음 청크 앞에 다시 넣기 |

**Recursive** 는 "큰 경계부터 차례로 시도한다"는 뜻입니다.
문단(`\n\n`) → 줄(`\n`) → 공백 → 글자 순으로 자를 곳을 찾습니다.
그래서 문장 중간이 잘리는 일이 줄어듭니다 (`chunk_basic_ko1.ipynb` 의 `chunk_paragraphs` 와 같은 생각).

`split_text` 와 `split_documents` 는 다릅니다.

- `split_text(문자열)` → **문자열 리스트**를 돌려줍니다 ← 여기서 쓰는 것
- `split_documents(Document 리스트)` → `Document` 리스트를 돌려줍니다

> **이 `chunks` 를 Step 7 에서 그대로 씁니다.** 여기서 어떻게 자르느냐가
> 검색 품질을 좌우합니다. Step 18 에서 그 영향을 직접 재 봅니다.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_text(book)      # split_text, not split_documents

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 191 chunks
First chunk:

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”


## Step 5. 임베딩 모델 — 내 PC 에서 돌린다

```python
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
```

여기가 `rag_basic_ko.ipynb` 와 가장 크게 다른 점입니다.

| | OpenAI 임베딩 | 이 노트북 (HuggingFace) |
|---|---|---|
| 도는 곳 | OpenAI 서버 | **내 PC** |
| 비용 | 토큰당 과금 | **공짜** |
| 데이터 | 외부로 나감 | **밖으로 안 나감** |
| 속도 | 빠름 | PC 사양에 달림 |
| 차원 | 1536 (`text-embedding-3-small`) | **384** |

`1_3_ollama.ipynb` 에서 본 "공짜·비공개를 얻는 대신 품질을 내준다"는 거래가
임베딩에서도 똑같이 적용됩니다.

> 처음 실행하면 모델(약 90MB)을 내려받습니다. 두 번째부터는 캐시를 씁니다.
> `all-MiniLM-L6-v2` 는 **공개 모델이라 로그인이 필요 없습니다.**
> 비공개 모델을 받을 때만 HuggingFace 토큰이 필요합니다.
> `all-MiniLM-L6-v2` 는 **영어 전용**에 가깝습니다. 한글 문서라면 다른 모델을 골라야 합니다.

In [9]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5579.48it/s]


## Step 6. 기존 DB 비우기

같은 폴더에 DB 가 남아 있으면 **이전 데이터에 새 데이터가 덧쌓입니다.**
그러면 검색 결과가 뒤섞여 실험을 다시 할 수 없습니다. 그래서 먼저 지웁니다.

`chunk_basic_ko2.ipynb` 에서도 같은 처리를 합니다.

In [10]:
import os
chroma_db = "chroma_vector_db"
if os.path.exists(chroma_db):
    Chroma(persist_directory=chroma_db, embedding_function=embeddings).delete_collection()

## Step 7. 벡터 DB 만들기 — 흔한 함정이 숨어 있는 자리

```python
vectorstore = Chroma.from_texts(texts=chunks, embedding=embeddings, persist_directory=chroma_db)
```

`texts=` 에 **Step 4 에서 만든 `chunks`(리스트)** 를 넘깁니다.
출력에 찍히는 숫자가 **청크 개수(약 190개)** 인지 반드시 확인하세요.

### 여기서 자주 나는 실수

`chunks` 대신 `book` 을 넘기면 어떻게 될까요? 오류는 **나지 않습니다.**

```python
Chroma.from_texts(texts=book, ...)     # book 은 문자열!
```

파이썬은 문자열을 순회하면 **글자를 하나씩** 내놓습니다.

```python
list("abc")   ->  ['a', 'b', 'c']
```

그래서 Chroma 가 **글자 하나를 문서 하나로** 14만 개를 만듭니다.
실제로 확인한 결과입니다.

```
book = "Alice was beginning to get very tired."   (38자)

texts=book   (문자열) -> 문서  38개   검색 결과: 'w', 'a', 't'
texts=chunks (리스트) -> 문서   2개   검색 결과: 'to get very tired.'
```

문법이 틀린 게 없으니 파이썬은 시키는 대로 할 뿐입니다.
**조용히 잘못 도는 코드**가 터지는 코드보다 찾기 어려운 이유입니다.

> 그래서 다음 Step 들에서 **검색 결과를 직접 찍어 보는 셀**(Step 14)이 꼭 필요합니다.

In [11]:
vectorstore = Chroma.from_texts(texts=chunks, embedding=embeddings, persist_directory=chroma_db)
print(f"Vectorstore created with {vectorstore._collection.count()} chunks")

Vectorstore created with 191 chunks


## Step 8. 챗봇 쪽 준비

여기서부터는 **질문에 답하는 부분**입니다. 앞에서 만든 DB 를 가져다 씁니다.

`gradio` 가 새로 들어옵니다. `1_6_2_stateful.ipynb` 에서 쓴 것과 같습니다.

In [12]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

## Step 9. 설정값

`gpt-4.1-nano` 는 OpenAI 의 가장 작고 싼 모델입니다.
답변 생성에만 쓰이고, **검색은 로컬 임베딩이 담당**합니다.

이렇게 섞어 쓰면 비용이 크게 줄어듭니다. 임베딩은 문서 전체에 대해 한 번 돌지만,
LLM 은 질문할 때만 부르기 때문입니다.

In [13]:
MODEL = "gpt-4.1-nano"
DB_NAME = "chroma_vector_db"
load_dotenv(override=True)

True

## Step 10. DB 다시 열기

**중요** — 저장할 때와 **같은 임베딩 모델**을 지정해야 합니다.

벡터는 모델마다 차원도 의미도 다릅니다. 384차원으로 저장해 놓고 1536차원 모델로 검색하면
맞지 않거나, 더 나쁘게는 **엉뚱한 결과가 조용히** 나옵니다.

`security_rag` 프로젝트가 DB 를 열 때 **백엔드 일치 검사**를 하는 이유가 이것입니다.

In [14]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2712.90it/s]


## Step 11. retriever 와 LLM

```python
retriever = vectorstore.as_retriever()
openai_llm = ChatOpenAI(temperature=0, model_name=MODEL)
```

`as_retriever()` 는 벡터 저장소를 **"질문을 주면 비슷한 문서를 돌려주는 물건"** 으로 감쌉니다.
기본값은 상위 4개입니다. 바꾸려면 이렇게 합니다.

```python
vectorstore.as_retriever(search_kwargs={"k": 3})
```

`temperature=0` 은 `1_1_2_temp_maxtoken.ipynb` 에서 배운 그대로입니다.
같은 질문에 같은 답이 나와야 결과를 비교할 수 있습니다.

In [15]:
retriever = vectorstore.as_retriever()
openai_llm = ChatOpenAI(temperature=0, model_name=MODEL)

## Step 12. 시스템 프롬프트

```
If you don't know the answer, say so.
Context:
{context}
```

두 가지가 들어 있습니다.

1. **역할** — "앨리스 이야기를 들려주는 사람"
2. **모르면 모른다고 하라** — 환각을 막는 장치 (`rag_basic_ko.ipynb` 와 같은 생각)

`{context}` 자리에 **검색해 온 청크들**이 들어갑니다.
RAG 의 G(생성)는 결국 이 한 칸을 채우는 일입니다.

> 이 칸에 무엇이 들어가느냐가 답을 결정합니다.
> Step 18 에서 **같은 질문인데 이 칸의 내용이 달라져 답이 뒤집히는** 것을 봅니다.

In [16]:
SYSTEM_PROMPT_TEMPLATE = """
You are a story teller on "Alice’s Adventures in Wonderland."
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

## Step 13. 질문에 답하는 함수

```python
docs = retriever.invoke(question)                         # ① 검색
context = "\n\n".join(doc.page_content for doc in docs)   # ② 이어 붙이기
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
response = openai_llm.invoke([SystemMessage(...), HumanMessage(...)])
```

RAG 의 전부가 이 네 줄입니다. `rag_basic_ko.ipynb` 에서 본 것과 같습니다.

> `1_1_openai.ipynb` 의 결론 **"messages 가 전부다"** 가 여기서도 그대로입니다.
> RAG 는 `system` 메시지에 무엇을 넣을지 자동으로 고르는 기술일 뿐입니다.

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = openai_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

## Step 14. 검색 결과를 직접 확인하기

```python
retriever.invoke("Who is Alice")
```

**답변을 보기 전에 검색 결과부터 봐야 합니다.**

여기서 `page_content` 에 **문장이 들어 있는지** 확인하세요.
공백이나 글자 한 개가 돌아온다면 Step 7 에서 `texts=` 를 잘못 넘긴 것입니다.

> 이 셀이 없으면 RAG 가 망가져도 알 수 없습니다.
> 다음 두 Step 에서 그 이유가 분명해집니다.

In [18]:
retriever.invoke("Who is Alice")

[Document(id='394c7c7a-3f69-4d22-8b4c-97b9f0693237', metadata={}, page_content='Alice was rather doubtful whether she ought not to lie down on her face\nlike the three gardeners, but she could not remember ever having heard\nof such a rule at processions; “and besides, what would be the use of a\nprocession,” thought she, “if people had all to lie down upon their\nfaces, so that they couldn’t see it?” So she stood still where she was,\nand waited.\n\nWhen the procession came opposite to Alice, they all stopped and looked\nat her, and the Queen said severely “Who is this?” She said it to the\nKnave of Hearts, who only bowed and smiled in reply.\n\n“Idiot!” said the Queen, tossing her head impatiently; and, turning to\nAlice, she went on, “What’s your name, child?”\n\n“My name is Alice, so please your Majesty,” said Alice very politely;\nbut she added, to herself, “Why, they’re only a pack of cards, after\nall. I needn’t be afraid of them!”'),
 Document(id='4b68f3b8-c6a0-4dfa-b051-ba9ff3

## Step 15. RAG 없이 물어보기

```python
openai_llm.invoke("Who is Alice")
```

문맥 없이 LLM 에게 직접 물었습니다. 보통 이런 답이 나옵니다.

> "Alice is a common given name for females... Without additional context,
> I can't determine which Alice you mean."

**"어느 앨리스인지 모르겠다"** 고 합니다. 당연합니다. 책 이야기라고 알려 주지 않았으니까요.

이것이 비교 기준(baseline)입니다.

In [19]:
openai_llm.invoke("Who is Alice")

AIMessage(content='Alice is a common given name for females, originating from Old French and Old German roots. It has been popularized through literature, most notably in Lewis Carroll\'s "Alice\'s Adventures in Wonderland," where Alice is the curious and imaginative young girl who explores a fantastical world. If you are referring to a specific person named Alice, please provide more context so I can assist you better.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 10, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_db27b144de', 'id': 'chatcmpl-EY9LariL1p46lt

## Step 16. RAG 로 물어보기 — 그런데 조심하세요

```python
answer_question("Who is Alice?", [])
```

이번에는 앨리스가 **책의 주인공**이라고 정확히 답합니다.

그런데 이 답이 **검색 덕분인지** 어떻게 알 수 있을까요?

| | Step 15 (RAG 없음) | Step 16 (RAG) |
|---|---|---|
| 검색 결과 | 없음 | 있음 |
| **시스템 프롬프트** | 없음 | **"앨리스 이야기를 들려주는 사람"** |

**두 가지가 동시에 달라졌습니다.** 답이 좋아진 것이 검색 때문인지,
시스템 프롬프트 한 줄 때문인지 이 비교만으로는 가릴 수 없습니다.

> **앨리스는 너무 유명합니다.**
> 모델이 이미 책 내용을 알고 있어서, 검색이 망가져도 그럴듯하게 답할 수 있습니다.
> **RAG 가 진짜 동작하는지 보려면 모델이 모를 만한 문서로 시험해야 합니다.**
> `rag_basic_ko.ipynb` 가 "가상의 회사 사내 규정"을 쓴 이유가 바로 이것입니다.

다음 Step 에서는 **모델의 지식으로는 설명되지 않는** 방식으로 검색의 영향을 봅니다.

In [20]:
answer_question("Who is Alice?", [])

'Alice is the main character in "Alice’s Adventures in Wonderland." She is a young girl who finds herself in a fantastical world filled with curious creatures and strange happenings. In the story, she is curious, polite, and often questions the bizarre events around her, trying to make sense of the whimsical world she has entered.'

## Step 17. Gradio 챗봇

```python
gr.ChatInterface(answer_question).launch()
```

함수 하나만 주면 채팅 화면이 만들어집니다 (`1_6_2_stateful.ipynb`).

`answer_question(message, history)` 의 두 번째 인자 `history` 를 **쓰지 않고 있습니다.**
즉 이 챗봇은 **지난 대화를 기억하지 못합니다** (`1_6_stateless.ipynb` 와 같은 상태).

기억하게 하려면 이렇게 고칩니다.

```python
def answer_question(question, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    messages = [SystemMessage(content=SYSTEM_PROMPT_TEMPLATE.format(context=context))]
    messages += history                                    # <- 이 줄 추가
    messages.append(HumanMessage(content=question))
    return openai_llm.invoke(messages).content
```

> 끝나면 셀을 중지하거나 `.close()` 로 서버를 내리세요.
> 그냥 두면 포트 7860 을 계속 잡고 있습니다.

In [17]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


---

# Step 18. 청크가 답을 자르면 — 같은 질문, 다른 답

지금까지는 청크 크기를 1,000자로 두고 넘어갔습니다.
이 값이 **왜 중요한지** 직접 확인합니다.

## 실험 설계

**모든 조건을 같게 두고 청크 크기만 바꿉니다.**

| | 같게 두는 것 | 바꾸는 것 |
|---|---|---|
| | 책 · 질문 · 임베딩 모델 · LLM · 프롬프트 | **`chunk_size` 와 `chunk_overlap`** |

| 설정 | 의도 |
|---|---|
| `chunk_size=300, overlap=0` | 작게 자르고 겹침 없음 — **경계에서 잘릴 위험이 크다** |
| `chunk_size=1000, overlap=200` | 노트북의 기본값 |

## 쓸 질문

책에 이런 대목이 있습니다 (모자 장수의 다과회).

```
"Once upon a time there were three little sisters," the Dormouse began...
"and their names were Elsie, Lacie, and Tillie; and they lived at the bottom of a well—"
"What did they live on?" said Alice...
"They lived on treacle," said the Dormouse...
```

여기에 두 가지를 묻습니다.

| 질문 | 정답 | 노리는 것 |
|---|---|---|
| **What did the three little sisters live on?** | treacle | 질문과 답이 **다른 청크로 갈린다** |
| Where did the three little sisters live? | a well | 질문과 답이 **같은 청크 안에 있다** (대조군) |

두 번째가 대조군입니다. 작은 청크가 **무조건** 나쁜 게 아니라
**답이 경계를 넘을 때만** 문제가 된다는 것을 보기 위해서입니다.

In [21]:
# 청크 크기만 바꿔 인덱스 두 개를 만듭니다. (책은 Step 3 에서 받아 둔 book 을 그대로 씁니다)
from langchain_text_splitters import RecursiveCharacterTextSplitter

SETTINGS = [(300, 0), (1000, 200)]
INDEXES = {}

for size, ov in SETTINGS:
    ch = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=ov).split_text(book)
    INDEXES[(size, ov)] = (
        ch,
        Chroma.from_texts(texts=ch, embedding=embeddings,
                          collection_name=f"demo-{size}-{ov}"),
    )
    print(f"chunk_size={size:<5} overlap={ov:<4} -> 청크 {len(ch):>4}개")

chunk_size=300   overlap=0    -> 청크  665개
chunk_size=1000  overlap=200  -> 청크  191개


In [22]:
# 정답이 들어 있는 대목이 어느 청크에 들어갔는지 눈으로 확인합니다.
for (size, ov), (ch, _) in INDEXES.items():
    print("=" * 76)
    print(f"chunk_size={size}, overlap={ov}")
    names = [i for i, c in enumerate(ch) if "Elsie" in c]        # 세 자매 이름
    answer = [i for i, c in enumerate(ch) if "lived on treacle" in c]  # 정답
    print(f"  '세 자매 이름'이 있는 청크 : {names}")
    print(f"  '정답(treacle)'이 있는 청크 : {answer}")
    print(f"  같은 청크 안에 있나         : {bool(set(names) & set(answer))}")
    if names:
        print(f"\n  이름이 있는 청크 내용:\n  {' '.join(ch[names[0]].split())[:300]}")
    print()

chunk_size=300, overlap=0
  '세 자매 이름'이 있는 청크 : [364]
  '정답(treacle)'이 있는 청크 : [365]
  같은 청크 안에 있나         : False

  이름이 있는 청크 내용:
  “Once upon a time there were three little sisters,” the Dormouse began in a great hurry; “and their names were Elsie, Lacie, and Tillie; and they lived at the bottom of a well—” “What did they live on?” said Alice, who always took a great interest in questions of eating and drinking.

chunk_size=1000, overlap=200
  '세 자매 이름'이 있는 청크 : [107]
  '정답(treacle)'이 있는 청크 : [107]
  같은 청크 안에 있나         : True

  이름이 있는 청크 내용:
  “Tell us a story!” said the March Hare. “Yes, please do!” pleaded Alice. “And be quick about it,” added the Hatter, “or you’ll be asleep again before it’s done.” “Once upon a time there were three little sisters,” the Dormouse began in a great hurry; “and their names were Elsie, Lacie, and Tillie; a



In [23]:
# 같은 질문을 두 인덱스에 던져 비교합니다. 프롬프트·모델은 위와 동일합니다.
QUESTIONS = [
    ("What did the three little sisters live on?", "treacle"),   # 답이 경계를 넘는다
    ("Where did the three little sisters live?", "well"),        # 대조군
]

for question, key in QUESTIONS:
    print("#" * 76)
    print(f"Q: {question}        (정답 낱말: '{key}')")
    for (size, ov), (_, store) in INDEXES.items():
        docs = store.similarity_search(question, k=2)
        context = "\n\n".join(d.page_content for d in docs)
        answer = openai_llm.invoke([
            SystemMessage(content=SYSTEM_PROMPT_TEMPLATE.format(context=context)),
            HumanMessage(content=question)]).content
        print(f"\n  --- chunk_size={size}, overlap={ov} ---")
        print(f"  검색 결과에 '{key}' 포함 : {key in context}")
        print(f"  답변: {answer}")
    print()

############################################################################
Q: What did the three little sisters live on?        (정답 낱말: 'treacle')

  --- chunk_size=300, overlap=0 ---
  검색 결과에 'treacle' 포함 : False
  답변: The story does not specify what the three little sisters—Elsie, Lacie, and Tillie—lived on. The Dormouse only begins to tell their story and mentions that they lived at the bottom of a well, but it doesn't say what they ate or drank.

  --- chunk_size=1000, overlap=200 ---
  검색 결과에 'treacle' 포함 : True
  답변: The three little sisters lived on treacle.

############################################################################
Q: Where did the three little sisters live?        (정답 낱말: 'well')

  --- chunk_size=300, overlap=0 ---
  검색 결과에 'well' 포함 : True
  답변: The three little sisters lived at the bottom of a well.

  --- chunk_size=1000, overlap=200 ---
  검색 결과에 'well' 포함 : True
  답변: The three little sisters lived at the bottom of a well.



## 결과 읽기

실행 결과를 보면 이렇게 나옵니다.

**질문 1 — "What did the three little sisters live on?" (정답: treacle)**

| 청크 설정 | 검색에 `treacle` 포함 | 답변 |
|---|---|---|
| `300 / 0` | **없음** | "책에 답이 없습니다" ← **틀렸습니다** |
| `1000 / 200` | 있음 | "treacle 을 먹고 살았습니다" |

**질문 2 — "Where did the three little sisters live?" (정답: well) · 대조군**

| 청크 설정 | 답변 |
|---|---|
| `300 / 0` | "우물 바닥에 살았습니다" |
| `1000 / 200` | "우물 바닥에 살았습니다" |

**둘 다 맞혔습니다.**

### 무슨 일이 일어났나

`chunk_size=300` 으로 자르면 이렇게 끊깁니다.

```
[청크 N  ] ...their names were Elsie, Lacie, and Tillie; and they lived at the
           bottom of a well—" "What did they live on?" said Alice...
                                 ↑ 질문까지만 들어 있다

[청크 N+1] "They lived on treacle," said the Dormouse...
                            ↑ 답은 다음 청크로 넘어갔다
```

질문 1 은 **"세 자매"와 "live on"** 이 들어간 청크 N 이 검색 1위가 됩니다.
의미상 가장 가까우니 당연합니다. 그런데 **그 청크에는 답이 없습니다.**

질문 2 는 "우물"이 청크 N 안에 들어 있어서 작은 청크로도 맞혔습니다.

> **작은 청크가 나쁜 것이 아닙니다.**
> **답이 청크 경계를 넘어갈 때만** 문제가 됩니다.
> 그리고 어느 질문이 경계를 넘을지는 **미리 알 수 없습니다.**

### 그래서 `chunk_overlap` 이 있습니다

겹침은 앞 청크의 끝부분을 다음 청크 앞에 **다시 넣습니다.**
그래서 경계에 걸친 내용이 **어느 한쪽에는 온전히** 들어가게 됩니다.

대가는 중복입니다. 청크가 늘고, 벡터가 늘고, 저장 비용이 늡니다.
`chunk_basic_ko1.ipynb` 에서 숫자로 확인했던 그 거래입니다.

### 기억할 것

| | |
|---|---|
| 청크 크기는 **검색 품질을 직접 바꾼다** | 모델도 프롬프트도 그대로인데 답이 뒤집혔습니다 |
| 질문과 답이 갈리면 못 찾는다 | 검색은 "질문과 비슷한 청크"를 찾지, "답이 든 청크"를 찾지 않습니다 |
| 겹침은 보험이다 | 경계에 걸친 답을 살려 줍니다. 대가는 중복 |
| **답변만 보면 모른다** | 질문 1 의 "답이 없습니다"는 그럴듯했지만 틀렸습니다 |

> 그렇다면 청크 크기는 몇이 좋을까요? **문서와 질문에 따라 다릅니다.**
> 정하는 방법이 아니라 **재는 방법**이 필요합니다.
> `chunk_basic_ko4.ipynb` 에서 채점표를 만들어 숫자로 고릅니다.

---

## 정리

### 흐름

| 단계 | 하는 일 | 쓴 도구 |
|---|---|---|
| 청킹 | 책을 1,000자 단위로 자르기 | `RecursiveCharacterTextSplitter` |
| 임베딩 | 글을 384차원 벡터로 — **내 PC 에서** | `HuggingFaceEmbeddings` |
| 저장 | 벡터 DB 에 보관 | `Chroma` |
| 검색 | 질문과 비슷한 청크 찾기 | `retriever.invoke()` |
| 생성 | 청크를 `system` 에 넣어 질문 | `ChatOpenAI` |

### 기억할 것

| | |
|---|---|
| **검색 결과를 직접 본다** | 답이 맞아 보여도 검색은 망가져 있을 수 있습니다 (Step 14) |
| **유명한 문서로 시험하지 않는다** | 모델이 이미 알면 RAG 가 망가져도 답이 맞습니다 (Step 16) |
| **청크 크기가 답을 바꾼다** | 답이 경계를 넘으면 못 찾습니다 (Step 18) |
| 문자열은 리스트가 아니다 | `texts=` 에 문자열을 주면 글자 단위로 쪼개집니다 (Step 7) |
| 임베딩 모델을 맞춘다 | 저장할 때와 검색할 때가 같아야 합니다 (Step 10) |
| 로컬 임베딩은 공짜·비공개 | 대신 품질과 속도를 양보합니다 (Step 5) |

### 직접 해 보기

1. Step 18 에서 `chunk_overlap` 만 `0 → 100` 으로 바꿔 보세요.
   작은 청크로도 질문 1 을 맞히게 되나요?
2. `chunk_size` 를 `150` 으로 더 줄여 보세요. 대조군(질문 2)도 틀리기 시작하나요?
3. 여러분이 직접 **답이 경계를 넘을 만한 질문**을 하나 더 만들어 보세요.
4. 앨리스 대신 **모델이 모를 만한 문서**(사내 규정, 내 메모)로 바꿔 보세요.
   Step 16 의 애매함이 사라집니다.
5. `as_retriever(search_kwargs={"k": 4})` 로 늘리면 질문 1 이 해결되나요?
   (힌트 — 겹침 없이도 다음 청크가 함께 딸려올 수 있습니다)

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `chunk_basic_ko1` ~ `ko4` | 청킹과 검색 품질을 **숫자로 재는** 방법 |
| `security_rag/` | 실제 CVE 데이터로 만드는 4단계 파이프라인 |